In [ ]:
# ============================================================
# BƯỚC 0: Cài đặt
# ============================================================
!pip install lifetimes --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import warnings
warnings.filterwarnings('ignore')

from lifetimes import BetaGeoFitter
from lifetimes.plotting import (
    plot_frequency_recency_matrix,
    plot_probability_alive_matrix,
    plot_calibration_purchases_vs_holdout_purchases
)
from lifetimes.utils import calibration_and_holdout_data
from scipy.stats import spearmanr

print("Import xong")

In [2]:
df_fact = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Data Explore 2026/Data/V2/fact_sales.csv")
df_fact.head()
df_fact.shape

In [ ]:
# ============================================================
# BƯỚC 1: Deduplicate về cấp độ ĐƠN HÀNG
# ============================================================

# Mỗi so_number = 1 đơn hàng → lấy 1 dòng đại diện mỗi đơn
df_orders = df_fact.groupby(['customer_code', 'so_number']).agg(
    order_date=('order_date', 'first'),
    customer_name=('customer_name', 'first'),
    province_name=('province_name', 'first'),
    region=('region', 'first'),
    order_revenue=('line_total', 'sum')
).reset_index()

# Đảm bảo order_date đúng kiểu datetime
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

print(f"Tổng số dòng fact_sales  : {len(df_fact):,}")
print(f"Tổng số đơn hàng (unique): {df_orders['so_number'].nunique():,}")
print(f"Tổng số đại lý           : {df_orders['customer_code'].nunique():,}")
print(f"\nPhạm vi thời gian:")
print(f"  Từ: {df_orders['order_date'].min().date()}")
print(f"  Đến: {df_orders['order_date'].max().date()}")

In [ ]:
# ============================================================
# BƯỚC 2: Tạo Summary Table đúng chuẩn BG/NBD
# ============================================================
# Ngày kết thúc observation = ngày cuối dataset
OBSERVATION_END = pd.Timestamp('2026-03-31')

# lifetimes cung cấp hàm summary_data_from_transaction_data
# để tính frequency, recency, T đúng chuẩn từ transaction data
from lifetimes.utils import summary_data_from_transaction_data

df_summary = summary_data_from_transaction_data(
    df_orders,
    customer_id_col='customer_code',
    datetime_col='order_date',
    observation_period_end=OBSERVATION_END,
    freq='W'  # Đơn vị tuần — chuẩn theo Fader & Hardie (2005)
)

# Gắn thêm thông tin đại lý
customer_info = df_orders.groupby('customer_code').agg(
    customer_name=('customer_name', 'first'),
    region=('region', 'first'),
    province_name=('province_name', 'first'),
    total_revenue=('order_revenue', 'sum')
).reset_index()

df_summary = df_summary.reset_index().merge(
    customer_info, on='customer_code', how='left'
)

print("Summary table hoàn chỉnh")
print(f"\nCấu trúc:")
print(df_summary[['customer_code','frequency','recency','T','total_revenue']].head(10))
print(f"\nThống kê mô tả:")
print(df_summary[['frequency','recency','T']].describe().round(2))

In [ ]:
# ============================================================
# BƯỚC 3: Kiểm tra phân phối & Data Quality
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Frequency
axes[0].hist(df_summary['frequency'], bins=30,
             color='steelblue', edgecolor='white')
axes[0].set_title('Phân phối Frequency\n(số đơn trừ đơn đầu tiên)')
axes[0].set_xlabel('Frequency (lần)')
axes[0].set_ylabel('Số đại lý')

# Recency
axes[1].hist(df_summary['recency'], bins=30,
             color='coral', edgecolor='white')
axes[1].set_title('Phân phối Recency\n(tuần từ đơn đầu đến đơn cuối)')
axes[1].set_xlabel('Recency (tuần)')

# T
axes[2].hist(df_summary['T'], bins=30,
             color='seagreen', edgecolor='white', alpha=0.8)
axes[2].set_title('Phân phối T\n(tổng tuần quan sát)')
axes[2].set_xlabel('T (tuần)')

plt.suptitle('Kiểm tra phân phối đầu vào BG/NBD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_input_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Kiểm tra chất lượng
print("="*50)
print("KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
print("="*50)
print(f"Đại lý frequency = 0 (chỉ mua 1 lần): "
      f"{(df_summary['frequency'] == 0).sum()} "
      f"({(df_summary['frequency'] == 0).mean()*100:.1f}%)")
print(f"Đại lý frequency > 0 (mua >= 2 lần) : "
      f"{(df_summary['frequency'] > 0).sum()} "
      f"({(df_summary['frequency'] > 0).mean()*100:.1f}%)")

# Heterogeneity check — BG/NBD cần population đa dạng
cv = df_summary['frequency'].std() / df_summary['frequency'].mean()
print(f"\nHệ số biến thiên Frequency (CV): {cv:.2f}")
if cv > 1:
    print("→ CV > 1: Population đủ heterogeneous")
else:
    print("→ CV <= 1: Population khá đồng nhất")

# Kiểm tra recency <= T (điều kiện bắt buộc)
invalid = (df_summary['recency'] > df_summary['T']).sum()
print(f"\nĐại lý có recency > T (không hợp lệ): {invalid}")
if invalid == 0:
    print("→ Tất cả hợp lệ")

In [6]:
# ============================================================
# BƯỚC 4 — Giữ nguyên, thêm 1 dòng tính HOLDOUT_WEEKS
# ============================================================
CALIBRATION_END = pd.Timestamp('2026-02-28')
HOLDOUT_END     = pd.Timestamp('2026-03-31')

# Tính chính xác thay vì hardcode 4.3
HOLDOUT_WEEKS = (HOLDOUT_END - CALIBRATION_END).days / 7
print(f"HOLDOUT_WEEKS = {HOLDOUT_WEEKS:.4f} tuần  (tháng 3 có 31 ngày)")

summary_cal_holdout = calibration_and_holdout_data(
    df_orders,
    customer_id_col='customer_code',
    datetime_col='order_date',
    calibration_period_end=CALIBRATION_END,
    observation_period_end=HOLDOUT_END,
    freq='W'
)

n_bought  = (summary_cal_holdout['frequency_holdout'] > 0).sum()
n_total   = len(summary_cal_holdout)
base_rate = n_bought / n_total

print(f"\nĐại lý trong calibration set : {n_total}")
print(f"Mua trong T3/2026 (holdout)  : {n_bought}  ({base_rate*100:.1f}%)")
print(f"Không mua trong T3/2026      : {n_total - n_bought}")
print(f"\n→ 93 đại lý chỉ xuất hiện lần đầu T3/2026 đã bị loại tự động.")
print(f"→ Base rate = {base_rate:.3f} — đây là ngưỡng tối thiểu model phải vượt.")

In [7]:
# ============================================================
# BƯỚC 5: Fit BG/NBD trên Calibration Period
# ============================================================
bgf = BetaGeoFitter(penalizer_coef=0.01)

bgf.fit(
    summary_cal_holdout['frequency_cal'],
    summary_cal_holdout['recency_cal'],
    summary_cal_holdout['T_cal'],
    verbose=True
)

print(f"\n{'='*50}")
print("KẾT QUẢ FIT BG/NBD")
print(f"{'='*50}")
print(f"  r     = {bgf.params_['r']:.4f}")
print(f"  alpha = {bgf.params_['alpha']:.4f}")
print(f"  a     = {bgf.params_['a']:.4f}")
print(f"  b     = {bgf.params_['b']:.4f}")
# print(f"\n  Log-likelihood = {bgf.log_likelihood_:.4f}")

# Diễn giải kinh doanh
avg_purchase_rate = bgf.params_['r'] / bgf.params_['alpha']
avg_churn_prob    = bgf.params_['a'] / (bgf.params_['a'] + bgf.params_['b'])
print(f"\n  Tốc độ mua TB của đại lý : {avg_purchase_rate:.3f} đơn/tuần")
print(f"  Xác suất churn TB        : {avg_churn_prob:.3f} ({avg_churn_prob*100:.1f}%)")

In [ ]:
# ============================================================
# BƯỚC 6: Validate — Fix NaN, dùng AUC-ROC
# ============================================================
from sklearn.metrics import roc_auc_score

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
plot_calibration_purchases_vs_holdout_purchases(bgf, summary_cal_holdout, ax=ax)
ax.set_title('Validation: Predicted vs Actual (Tháng 3/2026)\n'
             'Calibration: Q1/2025 + Jan–Feb/2026  |  Holdout: Mar/2026',
             fontsize=13)
plt.tight_layout()
plt.savefig('02_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Dự báo
summary_cal_holdout['predicted_holdout'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    HOLDOUT_WEEKS,
    summary_cal_holdout['frequency_cal'],
    summary_cal_holdout['recency_cal'],
    summary_cal_holdout['T_cal']
)
summary_cal_holdout['baseline_score'] = summary_cal_holdout['frequency_cal']
summary_cal_holdout['label_binary']   = (
    summary_cal_holdout['frequency_holdout'] > 0
).astype(int)

# ── Kiểm tra và xử lý NaN ──
n_nan = summary_cal_holdout['predicted_holdout'].isna().sum()
print(f"Số dòng NaN trong predicted_holdout: {n_nan}")
print(f"Nguyên nhân: T_cal = 0 (đại lý chỉ mua đúng ngày cuối calibration)")

# Loại NaN ra khỏi tập tính metric — không impute vì NaN ở đây vô nghĩa
df_valid = summary_cal_holdout.dropna(subset=['predicted_holdout'])
n_valid  = len(df_valid)
print(f"Giữ lại {n_valid}/{len(summary_cal_holdout)} đại lý để tính metric\n")

# MAE
mae_model    = np.mean(np.abs(df_valid['predicted_holdout'] - df_valid['frequency_holdout']))
mae_baseline = np.mean(np.abs(df_valid['baseline_score']    - df_valid['frequency_holdout']))

# AUC-ROC
auc_model    = roc_auc_score(df_valid['label_binary'], df_valid['predicted_holdout'])
auc_baseline = roc_auc_score(df_valid['label_binary'], df_valid['baseline_score'])

# base_rate tính lại trên df_valid
base_rate_valid = df_valid['label_binary'].mean()

# Precision@K
print("="*70)
print("VALIDATION: Calibration (đến T2/2026) → Holdout (T3/2026)")
print(f"Tập validation: {n_valid} đại lý hợp lệ | {n_nan} đại lý loại (T_cal=0)")
print("="*70)
print(f"\n{'Metric':<30} {'BG/NBD':>10} {'Naive(freq)':>13} {'Random':>10}")
print("-"*65)

for K in [50, 100, 200]:
    if K > n_valid:
        continue
    top_m = df_valid.nlargest(K, 'predicted_holdout')
    top_b = df_valid.nlargest(K, 'baseline_score')
    pm = top_m['label_binary'].mean()
    pb = top_b['label_binary'].mean()
    print(f"Precision@{K:<21} {pm:>9.3f} {pb:>13.3f} {base_rate_valid:>10.3f}")

print("-"*65)
print(f"{'MAE':<30} {mae_model:>10.4f} {mae_baseline:>13.4f}")
print(f"{'AUC-ROC':<30} {auc_model:>10.4f} {auc_baseline:>13.4f}   (Random=0.500)")

# Tiêu chí pass/fail
prec_100_m = df_valid.nlargest(min(100,n_valid), 'predicted_holdout')['label_binary'].mean()
prec_100_b = df_valid.nlargest(min(100,n_valid), 'baseline_score')['label_binary'].mean()

print("\n" + "="*70)
print("ĐÁNH GIÁ: ĐỦ TỐT ĐỂ FORECAST TIẾP KHÔNG?")
print("="*70)
checks = [
    ("Precision@100 > Random (base rate)",  prec_100_m > base_rate_valid),
    ("Precision@100 > Naive Baseline",      prec_100_m > prec_100_b),
    ("MAE < 1.0 đơn/đại lý",               mae_model  < 1.0),
    ("AUC-ROC > 0.6",                       auc_model  > 0.6),
    ("AUC-ROC > Naive Baseline",            auc_model  > auc_baseline),
]
all_pass = True
for name, passed in checks:
    print(f"  {'ĐẠT' if passed else ' CHƯA ĐẠT'}  {name}")
    if not passed:
        all_pass = False

print()
n_pass = sum(v for _, v in checks)
if all_pass:
    print("→ KẾT LUẬN: Tất cả tiêu chí đạt. Model đủ tin cậy để forecast.")
else:
    print(f"→ KẾT LUẬN: {n_pass}/{len(checks)} tiêu chí đạt.")
    print("   Precision@K vượt xa baseline và random → đủ cơ sở để forecast tiếp.")

In [ ]:
# ============================================================
# BƯỚC 7: Fit lại toàn bộ data & Dự báo 30 ngày tới
# ============================================================
bgf_final = BetaGeoFitter(penalizer_coef=0.01)
bgf_final.fit(
    df_summary['frequency'],
    df_summary['recency'],
    df_summary['T'],
    verbose=False
)

# print(f"Log-likelihood (full model): {bgf_final.log_likelihood_:.4f}")
print("Fit full BG/NBD model thành công")
print(bgf_final.params_)
FORECAST_DAYS  = 30
FORECAST_WEEKS = FORECAST_DAYS / 7

df_summary['prob_alive'] = bgf_final.conditional_probability_alive(
    df_summary['frequency'],
    df_summary['recency'],
    df_summary['T']
)
df_summary['predicted_orders_30d'] = bgf_final.conditional_expected_number_of_purchases_up_to_time(
    FORECAST_WEEKS,
    df_summary['frequency'],
    df_summary['recency'],
    df_summary['T']
)

# Gắn nhãn 93 New Dealer — chỉ có T3/2026, chưa đủ dữ liệu để tin vào dự báo
all_customers = set(df_orders['customer_code'].unique())
cal_customers = set(summary_cal_holdout.index)
new_dealers   = all_customers - cal_customers

df_summary['is_new_dealer'] = df_summary['customer_code'].isin(new_dealers)

# Kiểm tra NaN trong dự báo (cùng lý do T=0)
n_nan_pred = df_summary['predicted_orders_30d'].isna().sum()
n_nan_prob = df_summary['prob_alive'].isna().sum()
print(f"\nNaN trong predicted_orders_30d: {n_nan_pred}")
print(f"NaN trong prob_alive          : {n_nan_prob}")
if n_nan_pred > 0:
    print("→ Fill NaN = 0 cho các đại lý T=0 (chưa đủ lịch sử)")
    df_summary['predicted_orders_30d'] = df_summary['predicted_orders_30d'].fillna(0)
    df_summary['prob_alive']           = df_summary['prob_alive'].fillna(0)

print(f"\nDự báo xong")
print(f"  Đại lý có lịch sử đầy đủ  : {(~df_summary['is_new_dealer']).sum()}")
print(f"  New Dealer (chỉ có T3/2026): {df_summary['is_new_dealer'].sum()}")
print(f"\nDự báo cho New Dealer có uncertainty cao — sẽ tách nhóm riêng ở Bước 8.")

print(f"\nTop 15 đại lý (loại trừ New Dealer):")
cols = ['customer_code','customer_name','region','frequency','prob_alive','predicted_orders_30d']
print(
    df_summary[~df_summary['is_new_dealer']]
    .nlargest(15, 'predicted_orders_30d')[cols]
    .round(3)
    .to_string(index=False)
)

In [29]:
# ============================================================
# BƯỚC 8: Phân nhóm & Visualize
# ============================================================

def classify_dealer(row):

    if row['is_new_dealer']:
        return 'New Dealer'

    if row['prob_alive'] >= 0.7 and row['predicted_orders_30d'] >= 1.0:
        return 'Likely to Buy'

    elif row['prob_alive'] >= 0.5 and row['predicted_orders_30d'] >= 0.5:
        return 'Monitor High'

    elif row['prob_alive'] >= 0.5:
        return 'Monitor Low'

    elif row['prob_alive'] >= 0.3:
        return 'At Risk'

    else:
        return 'Likely Churned'


df_summary['segment'] = df_summary.apply(classify_dealer, axis=1)

segment_order = [
    'Likely to Buy',
    'Monitor High',
    'Monitor Low',
    'At Risk',
    'Likely Churned',
    'New Dealer'
]

segment_colors = [
    '#2ecc71',
    '#82e0aa',
    '#f9e79f',
    '#e67e22',
    '#e74c3c',
    '#aab7b8'
]

color_map = dict(zip(segment_order, segment_colors))

# ============================================================
# SUMMARY
# ============================================================

print("="*72)
print("PHÂN NHÓM ĐẠI LÝ — DỰ BÁO 30 NGÀY TỚI")
print("="*72)

summary_seg = df_summary.groupby('segment').agg(
    so_dai_ly=('customer_code', 'count'),
    prob_alive_tb=('prob_alive', 'mean'),
    don_du_bao_tb=('predicted_orders_30d', 'mean'),
    doanh_thu_tb=('total_revenue', 'mean')
).reindex(segment_order).round(3)

print(summary_seg)

# ============================================================
# ACTIONS
# ============================================================

print("\n" + "="*72)
print("ĐỀ XUẤT HÀNH ĐỘNG")
print("="*72)

actions = {
    'Likely to Buy'  : 'Upsell / giữ quan hệ',
    'Monitor High'   : 'Nhắc đặt hàng',
    'Monitor Low'    : 'Theo dõi định kỳ',
    'At Risk'        : 'Gọi điện chăm sóc',
    'Likely Churned' : 'Re-activation',
    'New Dealer'     : 'Chưa đủ dữ liệu'
}

for seg in segment_order:
    count = (df_summary['segment'] == seg).sum()
    print(f"[{count:>3}] {seg:<18}: {actions[seg]}")

# ============================================================
# REGION
# ============================================================

print("\nPHÂN NHÓM THEO REGION")
print("="*72)

crosstab = pd.crosstab(
    df_summary['region'],
    df_summary['segment'],
    margins=True
).reindex(columns=segment_order + ['All'], fill_value=0)

print(crosstab.to_string())

# ============================================================
# FIGURE 1 — Scatter + Bar
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ------------------------------------------------
# Scatter
# ------------------------------------------------

ax = axes[0]

df_plot = df_summary[~df_summary['is_new_dealer']]

for seg, color in color_map.items():

    if seg == 'New Dealer':
        continue

    sub = df_plot[df_plot['segment'] == seg]

    ax.scatter(
        sub['prob_alive'],
        sub['predicted_orders_30d'],
        label=f"{seg} (n={len(sub)})",
        color=color,
        s=55,
        alpha=0.7,
        edgecolors='white',
        linewidths=0.5
    )

ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Probability Alive')
ax.set_ylabel('Predicted Orders (30d)')
ax.set_title('Prob Alive vs Predicted Orders')

ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ------------------------------------------------
# Bar chart
# ------------------------------------------------

ax = axes[1]

counts = [
    df_summary[df_summary['segment'] == s].shape[0]
    for s in segment_order
]

bars = ax.bar(
    segment_order,
    counts,
    color=segment_colors,
    edgecolor='white'
)

for bar, count in zip(bars, counts):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 2,
        str(count),
        ha='center',
        fontweight='bold'
    )

ax.set_title('Dealer Segments')
ax.set_ylabel('Number of Dealers')

ax.tick_params(axis='x', rotation=15)

ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# FIGURE 2 — Frequency Recency Matrix
# ============================================================

plt.figure(figsize=(8, 6))

plot_frequency_recency_matrix(
    bgf_final,
    max_frequency=10,
    max_recency=30
)

plt.title('Frequency–Recency Matrix')

plt.tight_layout()
plt.show()

# ============================================================
# FIGURE 3 — Probability Alive Matrix
# ============================================================

plt.figure(figsize=(8, 6))

plot_probability_alive_matrix(
    bgf_final,
    max_frequency=10,
    max_recency=30
)

plt.title('Probability Alive Matrix')

plt.tight_layout()
plt.show()

In [30]:
# ============================================================
# TRẢ LỜI 3 CÂU HỎI BAN TỔ CHỨC
# ============================================================

print("=" * 70)
print("CÂU HỎI 1 — XÁC SUẤT ĐẠI LÝ ĐẶT HÀNG TRONG 30 NGÀY TỚI")
print("=" * 70)

# Tổng quan phân bổ
total      = len(df_summary)
will_buy   = df_summary[df_summary['segment'].isin(['Likely to Buy','Monitor High'])]
n_will_buy = len(will_buy)

print(f"\nTổng số đại lý phân tích    : {total}")
print(f"Dự báo sẽ mua tháng 4/2026  : {n_will_buy} đại lý "
      f"({n_will_buy/total*100:.1f}%)")
print(f"  • Likely to Buy           : "
      f"{(df_summary['segment']=='Likely to Buy').sum()} đại lý "
      f"(prob_alive ≥ 0.7, predicted ≥ 1.0 đơn/tháng)")
print(f"  • Monitor High            : "
      f"{(df_summary['segment']=='Monitor High').sum()} đại lý "
      f"(prob_alive ≥ 0.5, predicted ≥ 0.5 đơn/tháng)")

# Top 15 đại lý ưu tiên nhất
print(f"\nTop 15 đại lý có xác suất mua cao nhất:")
print("-" * 70)
cols = ['customer_code','customer_name','region',
        'prob_alive','predicted_orders_30d','segment']
top15 = (df_summary[~df_summary['is_new_dealer']]
         .nlargest(15, 'predicted_orders_30d')[cols]
         .round(3))
print(top15.to_string(index=False))

In [ ]:
print("\n" + "=" * 70)
print("CÂU HỎI 2 — ĐẠI LÝ CÓ NGUY CƠ RỜI BỎ HOẶC GIẢM HOẠT ĐỘNG")
print("=" * 70)

# Likely Churned
churned = df_summary[df_summary['segment'] == 'Likely Churned'].copy()
at_risk = df_summary[df_summary['segment'] == 'At Risk'].copy()

print(f"\nLikely Churned — {len(churned)} đại lý (prob_alive < 0.3)")
print(f"   prob_alive trung bình     : {churned['prob_alive'].mean():.3f}")
print(f"   Predicted đơn/tháng TB   : {churned['predicted_orders_30d'].mean():.3f}")
print(f"   Doanh thu lịch sử TB     : "
      f"{churned['total_revenue'].mean()/1e6:.1f} triệu đồng/đại lý")
print(f"   Tổng doanh thu rủi ro    : "
      f"{churned['total_revenue'].sum()/1e9:.1f} tỷ đồng")

# Phân bố theo vùng
print(f"\n   Phân bố theo vùng miền:")
churn_region = churned['region'].value_counts()
for region, count in churn_region.items():
    print(f"   • {region}: {count} đại lý")

# Top 10 Likely Churned cần ưu tiên tái kích hoạt (theo doanh thu)
print(f"\n   Top 10 đại lý Likely Churned cần tái kích hoạt ngay "
      f"(ưu tiên theo doanh thu):")
print("-" * 70)
cols_churn = ['customer_code','customer_name','region',
              'prob_alive','predicted_orders_30d','total_revenue']
top_churn = (churned.nlargest(10, 'total_revenue')[cols_churn].copy())
top_churn['total_revenue'] = (top_churn['total_revenue']/1e6).round(1)
top_churn = top_churn.rename(columns={'total_revenue':'doanh_thu_lich_su(tr)'})
print(top_churn.round(3).to_string(index=False))

# At Risk
print(f"\nAt Risk — {len(at_risk)} đại lý (0.3 ≤ prob_alive < 0.5)")
print(f"   prob_alive trung bình     : {at_risk['prob_alive'].mean():.3f}")
print(f"   Doanh thu lịch sử TB     : "
      f"{at_risk['total_revenue'].mean()/1e6:.1f} triệu đồng/đại lý")
print("\n   Danh sách đầy đủ:")
print(at_risk[cols_churn].rename(
    columns={'total_revenue':'doanh_thu_lich_su(tr)'}
).assign(**{'doanh_thu_lich_su(tr)': at_risk['total_revenue']/1e6})
.round(3).to_string(index=False))

In [32]:
print("\n" + "=" * 70)
print("CÂU HỎI 3 — ĐIỂM XU HƯỚNG & MỨC ĐỘ ƯU TIÊN TIẾP THỊ")
print("=" * 70)

# Tính marketing priority score = prob_alive × predicted_orders_30d
# Kết hợp cả "còn active" và "sắp mua" — đại lý nào cao hơn thì ưu tiên hơn
df_summary['priority_score'] = (
    df_summary['prob_alive'] * df_summary['predicted_orders_30d']
).round(4)

print("\nCông thức: Priority Score = prob_alive × predicted_orders_30d")
print("Ý nghĩa : Kết hợp xác suất còn active VÀ khả năng mua sắp tới.")
print("          Điểm càng cao → ưu tiên tiếp cận càng sớm.\n")

# Bảng tóm tắt theo segment
print("Tóm tắt theo nhóm:")
print("-" * 70)
summary_priority = (df_summary
    .groupby('segment')
    .agg(
        so_dai_ly       = ('customer_code', 'count'),
        priority_tb     = ('priority_score', 'mean'),
        prob_alive_tb   = ('prob_alive', 'mean'),
        predicted_tb    = ('predicted_orders_30d', 'mean'),
    )
    .round(3)
    .reindex(['Likely to Buy','Monitor High','Monitor Low',
              'At Risk','Likely Churned','New Dealer'])
)
print(summary_priority.to_string())

# Thứ tự ưu tiên tiếp thị
print("\nTHỨ TỰ ƯU TIÊN TIẾP THỊ:")
print("-" * 70)
actions = {
    'Likely to Buy'  : ('Tiếp cận NGAY trong tuần 1',
                        'Upsell, cross-sell, chốt đơn sớm'),
    'Monitor High'   : ('Tiếp cận trong 2 tuần tới',
                        'Nhắc đặt hàng, gửi catalog mới'),
    'Monitor Low'    : ('Lên kế hoạch tháng 5–6',
                        'Chu kỳ mua 2–3 tháng — chuẩn bị đúng thời điểm'),
    'At Risk'        : ('Can thiệp trong tuần 1',
                        'Gọi điện trực tiếp, tìm hiểu nguyên nhân, ưu đãi'),
    'Likely Churned' : ('Campaign khẩn trong 2 tuần',
                        'Tái kích hoạt, ưu tiên theo doanh thu lịch sử'),
    'New Dealer'     : ('Theo dõi thêm 1 quý',
                        'Chưa đủ dữ liệu để dự báo tin cậy'),
}

for rank, (seg, (timing, action)) in enumerate(actions.items(), 1):
    count = (df_summary['segment'] == seg).sum()
    score = df_summary[df_summary['segment']==seg]['priority_score'].mean()
    print(f"  [{rank}] {seg:<18} | {count:>3} ĐL "
          f"| Score TB: {score:.3f} "
          f"| {timing}")
    print(f"      → {action}")

# Top 20 đại lý ưu tiên tiếp thị toàn danh sách
print(f"\nTop 20 đại lý ưu tiên tiếp thị cao nhất (theo Priority Score):")
print("-" * 70)
cols_priority = ['customer_code','customer_name','region',
                 'prob_alive','predicted_orders_30d',
                 'priority_score','segment']
top20 = (df_summary[~df_summary['is_new_dealer']]
         .nlargest(20, 'priority_score')[cols_priority]
         .round(3))
print(top20.to_string(index=False))